# Discover a feature signature
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/v1/cookbook/notebooks/04_discover_feature_signature.ipynb)

Compare feature prevalence between positives and background, inspect logos, and export targets for RL-SAE. The small default compares activation domains with repression domains; it is a contrast between these groups, not a general activity classifier.

This notebook runs independently. Select **Runtime → Change runtime type → GPU** in Colab.
First use downloads model weights. A GPU is recommended; CPU inference is supported but slower. Reduce sample counts and batch size for a first run.
The setup installs the `v1` release when IDiom is absent. If using an older installation,
upgrade to that release and restart the kernel. No adjacent helper files are required.

In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("idiom") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "idiom[cookbook] @ git+https://github.com/rotskoff-group/idiom.git@v1"])
if importlib.util.find_spec("pandas") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas>=2"])

import json
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from idiom import IDiom, IDiomSAE
from idiom.data.records import Record
from idiom.utils.notebook_helpers import (
    load_inputs, idr_sequence, isolated, check_context, summaries, write_fasta,
    save_run, sequence_metrics, nearest_reference, split_records, example_file,
)
print("Python:", sys.version.split()[0])
started = time.perf_counter()

## Inputs and settings
Run top to bottom. Upload a FASTA using Colab's Files pane and set its path below, or
leave the demo input unchanged. `INPUT_MODE="idr"` accepts ordinary headers for isolated
IDRs; `"annotated"` requires full-protein headers ending in `_IDR_x-y` (1-based inclusive).
Python coordinates are 0-based, end-exclusive. These workflows do not predict IDR boundaries.

For persistent outputs, optionally mount Drive in your own cell with
`from google.colab import drive; drive.mount("/content/drive")`, then set `OUT_DIR` there.
Use a new output directory for each experiment. Rejected records are reported in an audit.

Choose a background that represents the alternative you care about. Length matching does not
control homology, composition, or assay differences. Exact duplicate IDRs are collapsed, and
all positive IDRs are excluded from the background before sampling.

In [ ]:
POSITIVE_FASTA = None # Default: activation-domain example FASTA
BACKGROUND_FASTA = None # Default: repression-domain example FASTA
POSITIVE_MODE = "idr"
BACKGROUND_MODE = "idr"
SAE_ID = "jxliu2/idiomsae-300M-L18-k32"
DEVICE = "auto"
MAX_POSITIVE = 32
MAX_BACKGROUND = 64
BATCH_SIZE = 2
SEED = 0
TOP_N = 30
NAME = "ad_vs_rd"
CASE = "top30"
FDR_ALPHA = 0.001
LOG2OR_FLOOR = 1.0
PREV_POS_FLOOR = 0.05
DROP_BOUNDARY = True
N_FEATURES = 6
N_WINDOWS = 30
HALF_WIDTH = 7
EXPORT_SIGNATURE = True
RESULT_DIR = None # Reopen a CLI/notebook run containing enrichment.npz and fd_positive
OUT_DIR = Path("enrichment_outputs")

## Prepare the comparison or reopen results

In [ ]:
from idiom.sae.features import (
    FeatureDataset, AMINO_ACIDS, logo_data, enrich, feature_counts, select_features,
    save_enrichment, load_enrichment, write_signature,
)
from idiom.sae.features.enrichment import length_match, prepare_sequences
if OUT_DIR.exists() and any(OUT_DIR.iterdir()):
    raise ValueError("Use a new or empty OUT_DIR; set RESULT_DIR to inspect a previous run.")
OUT_DIR.mkdir(parents=True, exist_ok=True)
if RESULT_DIR is None:
    positive_path = POSITIVE_FASTA or example_file("effector/ad.fasta", OUT_DIR / "inputs")
    background_path = BACKGROUND_FASTA or example_file("effector/rd.fasta", OUT_DIR / "inputs")
    sae = IDiomSAE.from_pretrained(SAE_ID, device=DEVICE)
    if sae.fim_mode != "unprompted" or sae.region != "idr":
        raise ValueError("Use an unprompted IDR SAE for this workflow.")

    def unique_inputs(path, mode):
        records, audit = load_inputs(path, mode, None)
        kept, reasons = prepare_sequences(records, max_length=sae.model.cfg.max_seq_len - 4)
        for record, reason in zip(records, reasons):
            if reason:
                audit.loc[audit.record_id == record.accession, "status"] = reason
        return kept, audit

    positive_pool, positive_audit = unique_inputs(positive_path, POSITIVE_MODE)
    background_pool, background_audit = unique_inputs(background_path, BACKGROUND_MODE)
    positive_idrs = {idr_sequence(r) for r in positive_pool}
    overlap = [r.accession for r in background_pool if idr_sequence(r) in positive_idrs]
    background_audit.loc[background_audit.record_id.isin(overlap), "status"] = "overlaps positives"
    background_pool = [r for r in background_pool if idr_sequence(r) not in positive_idrs]
    if MAX_POSITIVE < 1 or MAX_BACKGROUND < 1:
        raise ValueError("Sample limits must be positive.")
    chosen = np.random.default_rng(SEED).permutation(len(positive_pool))[:MAX_POSITIVE]
    positives = [positive_pool[i] for i in sorted(chosen)]
    if not positives or not background_pool:
        raise ValueError("Need nonempty positive and nonoverlapping background sets.")
    background = length_match(positives, background_pool, n=MAX_BACKGROUND, rng=np.random.default_rng(SEED))
    for label, records, audit in (("positive", positives, positive_audit), ("background", background, background_audit)):
        audit["selected"] = audit.record_id.isin([r.accession for r in records])
        audit.to_csv(OUT_DIR / f"{label}_input_audit.csv", index=False)
        write_fasta(records, OUT_DIR / f"{label}_selected.fasta")
        display(audit.groupby(["status", "selected"]).size())
    fig, ax = plt.subplots(figsize=(6, 3), constrained_layout=True)
    lengths = [[len(idr_sequence(r)) for r in group] for group in (positives, background)]
    bins = np.histogram_bin_edges(lengths[0] + lengths[1], bins=15)
    for values, label in zip(lengths, ("positive", "background")):
        ax.hist(values, bins=bins, density=True, histtype="step", label=label)
    ax.set(xlabel="IDR length", ylabel="Density")
    ax.legend()
    fig.savefig(OUT_DIR / "background_lengths.png", dpi=160)
    plt.show()
    pos_fd = sae.build_feature_dataset(positives, OUT_DIR / "fd_positive", batch_size=BATCH_SIZE)
    bg_fd = sae.build_feature_dataset(background, OUT_DIR / "fd_background", batch_size=BATCH_SIZE)
    a, n_pos = feature_counts(pos_fd)
    b, n_neg = feature_counts(bg_fd)
    result = enrich(a, n_pos, b, n_neg, sae.sae.num_latents)
else:
    result, _ = load_enrichment(Path(RESULT_DIR) / "enrichment.npz")
    pos_fd, bg_fd = Path(RESULT_DIR) / "fd_positive", Path(RESULT_DIR) / "fd_background"
    if not pos_fd.exists() or not bg_fd.exists():
        saved = json.loads((Path(RESULT_DIR) / "run.json").read_text())
        saved = saved.get("settings", saved)
        pos_fd, bg_fd = Path(saved["positive_dataset"]), Path(saved["background_dataset"])
dataset = FeatureDataset(pos_fd)
selection = select_features(result, n=TOP_N, drop_boundary=DROP_BOUNDARY, feature_dir=bg_fd,
                            fdr_alpha=FDR_ALPHA, log2or_floor=LOG2OR_FLOOR, prev_pos_floor=PREV_POS_FLOOR)
ids = selection["ids"]
save_enrichment(OUT_DIR / "enrichment.npz", result, selection)

## Inspect enrichment and exclusions
A feature counts once per sequence if any activation is positive. Statistics use smoothed log₂
odds ratios, a standardized count under a hypergeometric null, approximate normal p-values,
and Benjamini–Hochberg correction. Boundary filtering is a heuristic for terminal artifacts.
No selected features is a valid result, particularly with small samples.

In [ ]:
table = pd.DataFrame({key: value for key, value in result.items() if isinstance(value, np.ndarray)})
table.insert(0, "feature_id", range(len(table)))
for key in ("enriched", "boundary_filtered", "selected"):
    table[key] = selection[key]
table.to_csv(OUT_DIR / "enrichment.tsv", sep="\t", index=False)
display(table.loc[table.enriched].sort_values("log2or", ascending=False).head(20))
fig, ax = plt.subplots(figsize=(6, 3), constrained_layout=True)
ax.scatter(table.log2or, np.abs(table.z), c=table.selected.astype(int), cmap="coolwarm", s=8)
ax.set(xlabel="log₂ odds ratio", ylabel="|z|", title=f"{len(ids)} selected features")
fig.savefig(OUT_DIR / "enrichment.png", dpi=160)
plt.show()
feature_ids = ids[:N_FEATURES]

## Inspect selected features
Peak-centered windows use the same calculation as notebook 03. Missing edge positions remain aligned.

In [ ]:
import logomaker
window_rows = []
for feature in feature_ids:
    data = logo_data(dataset, feature, n=N_WINDOWS, half_width=HALF_WIDTH)
    fig, ax = plt.subplots(figsize=(7, 2), constrained_layout=True)
    if data["windows"]:
        logomaker.Logo(pd.DataFrame(data["information"], columns=list(AMINO_ACIDS)), ax=ax, color_scheme="chemistry")
        profile = data["mean_activation"]
        profile = profile / max(float(profile.max()), 1e-12)
        for i, value in enumerate(profile):
            ax.axvspan(i - 0.5, i + 0.5, color="orange", alpha=float(value) * 0.4, zorder=0)
    ax.axvline(HALF_WIDTH, color="black", linestyle="--", linewidth=0.5)
    ax.set_xticks(range(len(data["offsets"])), data["offsets"])
    ax.set(title=f"F{feature}: {len(data['windows'])} windows", xlabel="Offset from peak", ylabel="Bits")
    fig.savefig(OUT_DIR / f"feature_{feature}_logo.png", dpi=160)
    plt.show()
    np.savez(OUT_DIR / f"feature_{feature}_logo.npz", counts=data["counts"],
             mean_activation=data["mean_activation"], offsets=data["offsets"])
    window_rows.extend(dict(feature_id=feature, sequence_id=w.sequence_id,
                            peak_position=w.peak_position, window=w.residues) for w in data["windows"])
pd.DataFrame(window_rows, columns=["feature_id", "sequence_id", "peak_position", "window"]).to_csv(
    OUT_DIR / "feature_windows.csv", index=False)

## Export a signature
The JSON is directly usable by notebook 07 and the SAE GRPO script. Feature IDs are tied to
SAE weights. The saved dataset identifies the SAE used; legacy datasets without provenance
require you to supply the correct SAE_ID. A short example run need not yield a useful signature.

In [ ]:
recorded_sae = dataset.provenance.get("sae") or SAE_ID
provenance = dict(sae=recorded_sae, host_model=dataset.provenance.get("host_model"),
                  layer=dataset.layer, num_latents=dataset.num_latents, seed=SEED,
                  n_pos=result["n_pos"], n_background=result["n_neg"],
                  positive_dataset=str(Path(pos_fd).resolve()), background_dataset=str(Path(bg_fd).resolve()))
if EXPORT_SIGNATURE and ids:
    write_signature(OUT_DIR / "signature.json", {NAME: ids}, case=CASE, provenance=provenance)
    print("Signature:", NAME, "Case:", CASE, "Features:", ids)
else:
    print("No signature written. The complete enrichment results are saved.")
save_run(OUT_DIR, dict(**provenance, top_n=TOP_N, fdr_alpha=FDR_ALPHA, log2or_floor=LOG2OR_FLOOR,
                       prev_pos_floor=PREV_POS_FLOOR, drop_boundary=DROP_BOUNDARY),
         elapsed=time.perf_counter() - started)

## Download and continue
Open [RL-SAE design](07_design_with_rl_sae.ipynb) with the exported signature, or use its bundled target.

In [ ]:
import shutil
archive = shutil.make_archive(str(OUT_DIR.resolve()), "zip", OUT_DIR)
print("Results:", OUT_DIR.resolve(), "\nDownload:", archive)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(archive)